In [21]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
root_path = os.path.abspath(os.path.join(os.getcwd(), '../'))
sys.path.append(root_path)
import pandas as pd
import numpy as np
from core.data_sources.clob import CLOBDataSource
from core.data_structures.candles import Candles

In [22]:
root_path

'/Users/schalkvisagie/csHonours/project/25349589-MN8-src/quants-lab'

In [23]:
# Load Candles
clob = CLOBDataSource()
CONNECTOR_NAME = "binance"
INTERVALS = "1s"
trading_pair = "POL-USDT"
# DAYS = 60

clob.load_candles_cache(root_path)
all_candles = clob.get_candles_from_cache(CONNECTOR_NAME, trading_pair, INTERVALS)
print(all_candles)

candles: Candles = all_candles
candlesdf = candles.data

# candlesdf
# filtered_df = df[df['quote_asset_volume'] > 0]
# print(filtered_df)

2025-06-14 16:39:44,254 - asyncio - ERROR - Task was destroyed but it is pending!
task: <Task pending name='Task-5' coro=<safe_wrapper() running at /opt/homebrew/anaconda3/envs/quants-lab/lib/python3.12/site-packages/hummingbot/core/utils/async_utils.py:9> wait_for=<Future pending cb=[Task.task_wakeup()]>>
2025-06-14 16:39:44,278 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x17ff580b0>


In [24]:
def load_market_data(connector_name: str, trading_pair: str, data_type: str = "order_book") -> pd.DataFrame:
    """
    Load market data from files for a specific connector and trading pair.
    
    Args:
        connector_name: Name of the connector (e.g., "bitmart_paper_trade")
        trading_pair: Trading pair symbol (e.g., "LINK-USDT")
        data_type: Type of data to load ("order_book" or "trades")
    
    Returns:
        pd.DataFrame: Concatenated DataFrame containing all data from matching files
    """
    folder = root_path + f"/data/order_book/"
    
    # Define the pattern based on data type
    pattern = "order_book_snapshots" if data_type == "order_book" else "trades"
    
    # Find all matching files
    files = [
        file for file in os.listdir(folder) 
        if connector_name in file 
        and trading_pair in file 
        and pattern in file
    ]
    
    if not files:
        raise FileNotFoundError(f"No {data_type} files found for {connector_name} {trading_pair}")
    
    # Load and concatenate all matching files
    dfs = []
    for file in files:
        df = pd.read_json(folder + "/" + file, lines=True)
        dfs.append(df)
        print(file)
    
    return pd.concat(dfs, ignore_index=True)

# Example usage:
order_book_df = load_market_data(CONNECTOR_NAME, trading_pair, "order_book")


binance_POL-USDT_order_book_snapshots_2025-05-31.txt
binance_POL-USDT_order_book_snapshots_2025-06-14.txt


In [25]:
order_book_df.rename(columns={"ts": "timestamp"}, inplace=True)
order_book_df

,timestamp,bids,asks
0,1748699400,"[[0.2111, 8741.3], [0.211, 37744.6], [0.2109, ...","[[0.2112, 12250.1], [0.21130000000000002, 4998..."
1,1748699401,"[[0.2111, 9688.5], [0.211, 37744.6], [0.2109, ...","[[0.2112, 14207.3], [0.21130000000000002, 5420..."
2,1748699402,"[[0.2111, 18758.0], [0.211, 37744.6], [0.2109,...","[[0.2112, 9235.1], [0.21130000000000002, 51279..."
3,1748699403,"[[0.2111, 18758.0], [0.211, 37744.6], [0.2109,...","[[0.2112, 9235.1], [0.21130000000000002, 51279..."
4,1748699404,"[[0.2111, 18770.1], [0.211, 37744.6], [0.2109,...","[[0.2112, 9008.9], [0.21130000000000002, 42093..."
...,...,...,...
18346,1749911973,"[[0.2, 18340.2], [0.19990000000000002, 32608.8...","[[0.2001, 18275.1], [0.20020000000000002, 5648..."
18347,1749911974,"[[0.2, 12340.2], [0.19990000000000002, 31798.6...","[[0.2001, 18275.1], [0.20020000000000002, 6083..."
18348,1749911975,"[[0.2, 12340.2], [0.19990000000000002, 46804.2...","[[0.2001, 18275.1], [0.20020000000000002, 6083..."
18349,1749911976,"[[0.2, 12340.2], [0.19990000000000002, 46804.2...","[[0.2001, 18275.1], [0.20020000000000002, 6033..."


In [26]:
candlesdf
order_book_df

# Ensure timestamp columns are of the same type (int)
candlesdf['timestamp'] = candlesdf['timestamp'].astype(int)
order_book_df['timestamp'] = order_book_df['timestamp'].astype(int)

# Merge on 'timestamp'
candles_and_ob_df = pd.merge(
    candlesdf,
    order_book_df,
    on='timestamp',
    how='inner',  # Only keep rows with matching timestamps
    suffixes=('_candle', '_orderbook')
)

# Display the merged DataFrame
candles_and_ob_df['datetime'] = pd.to_datetime(candles_and_ob_df['timestamp'], unit='s')
# candles_and_ob_df.head()
candles_and_ob_df

,timestamp,open,high,low,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume,bids,asks,datetime
0,1748699400,0.2112,0.2112,0.2112,0.2112,0,0,0,0,0,"[[0.2111, 8741.3], [0.211, 37744.6], [0.2109, ...","[[0.2112, 12250.1], [0.21130000000000002, 4998...",2025-05-31 13:50:00
